
# PoC — US02: Consulta Automatizada de Normativa Urbana (RAG Agéntico)

**Proyecto:** Sistema Multiagente para la Priorización de Proyectos de Inversión en la Municipalidad Distrital de El Tambo

Esta notebook implementa una **Prueba de Concepto técnica** para la historia de usuario **US02**:

- recuperar normativa relevante;
- usar búsqueda **híbrida** (semántica + BM25);
- mostrar la evidencia recuperada;
- medir el rendimiento del RAG;
- decidir **GO / GO CON CONDICIONES / NO-GO**.

> **IMPORTANTE:** la base incluida es **didáctica y simulada**. No contiene citas oficiales ni debe utilizarse para emitir un dictamen legal real. Para la evidencia final del proyecto, reemplaza estos resúmenes por fragmentos de documentos oficiales vigentes.



## Antes de ejecutar

En Google Colab:

1. Abre **Entorno de ejecución → Cambiar tipo de entorno de ejecución**.
2. Selecciona **GPU T4** si está disponible.
3. Ejecuta las celdas de arriba hacia abajo.

La notebook compara:
- `BAAI/bge-m3`
- `Qwen/Qwen3-Embedding-0.6B`
- `text-embedding-3-small` **opcional**, si agregas una API key de OpenAI.

Para evitar costos, puedes completar la PoC únicamente con los dos modelos locales y dejar OpenAI como comparación opcional.


In [1]:

!pip -q install -U "sentence-transformers>=5.0.0" qdrant-client rank-bm25 pandas numpy scikit-learn openai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
numb

In [1]:

import os, time, re, math, getpass
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from rank_bm25 import BM25Okapi
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
print("Entorno listo.")


Entorno listo.



## 1. Base de datos de prueba generada

Esta base permite **probar el funcionamiento técnico** del RAG sin usar expedientes reales.  
Cada fila representa un fragmento normativo resumido y etiquetado.

Para el informe final, sustituye `contenido` por extractos o fragmentos procesados desde las **fuentes oficiales vigentes**.


In [2]:

records = [['LEY27972-01', 'Ley N.° 27972', 'Competencias municipales', 'Resumen didáctico simulado: las municipalidades ejercen competencias y funciones en asuntos de interés local dentro del marco legal correspondiente.', False], ['LEY27972-02', 'Ley N.° 27972', 'Desarrollo urbano', 'Resumen didáctico simulado: la gestión municipal comprende funciones vinculadas con organización del espacio físico, desarrollo urbano y ordenamiento local.', False], ['LEY27972-03', 'Ley N.° 27972', 'Concejo municipal', 'Resumen didáctico simulado: el concejo municipal cumple funciones normativas y fiscalizadoras dentro del gobierno local.', False], ['LEY27972-04', 'Ley N.° 27972', 'Alcaldía', 'Resumen didáctico simulado: la alcaldía ejerce funciones ejecutivas y de representación del gobierno local conforme a sus atribuciones.', False], ['LEY27972-05', 'Ley N.° 27972', 'Servicios públicos locales', 'Resumen didáctico simulado: la municipalidad participa en la organización y prestación de servicios públicos locales dentro de sus competencias.', False], ['LEY27972-06', 'Ley N.° 27972', 'Planeamiento y presupuesto', 'Resumen didáctico simulado: la gestión municipal articula instrumentos de planeamiento, presupuesto y programación para atender objetivos locales.', False], ['DL1252-01', 'D. L. N.° 1252 - Invierte.pe', 'Programación multianual', 'Resumen didáctico simulado: la programación multianual orienta la cartera de inversiones hacia el cierre de brechas y la priorización de necesidades públicas.', False], ['DL1252-02', 'D. L. N.° 1252 - Invierte.pe', 'Formulación y evaluación', 'Resumen didáctico simulado: los proyectos pasan por actividades de formulación y evaluación para sustentar su conveniencia y viabilidad antes de continuar.', False], ['DL1252-03', 'D. L. N.° 1252 - Invierte.pe', 'Ejecución de inversiones', 'Resumen didáctico simulado: la ejecución comprende el desarrollo de las inversiones aprobadas conforme a la programación, expediente y controles aplicables.', False], ['DL1252-04', 'D. L. N.° 1252 - Invierte.pe', 'Cierre de brechas', 'Resumen didáctico simulado: el sistema de inversiones públicas busca orientar recursos hacia brechas de infraestructura o acceso a servicios identificadas.', False], ['DL1252-05', 'D. L. N.° 1252 - Invierte.pe', 'Cartera de inversiones', 'Resumen didáctico simulado: la cartera agrupa inversiones priorizadas y permite organizar su programación de acuerdo con objetivos y disponibilidad de recursos.', False], ['DL1252-06', 'D. L. N.° 1252 - Invierte.pe', 'Seguimiento', 'Resumen didáctico simulado: el seguimiento de inversiones permite revisar avances, consistencia de información y cumplimiento de la programación.', False], ['LEY32069-01', 'Ley N.° 32069', 'Principios de contratación pública', 'Resumen didáctico simulado: la contratación pública debe seguir principios de integridad, competencia, transparencia y uso eficiente de recursos.', False], ['LEY32069-02', 'Ley N.° 32069', 'Planificación de contrataciones', 'Resumen didáctico simulado: las contrataciones deben planificarse de acuerdo con las necesidades de la entidad y la disponibilidad presupuestal.', False], ['LEY32069-03', 'Ley N.° 32069', 'Procedimientos de selección', 'Resumen didáctico simulado: la selección de proveedores se realiza mediante procedimientos definidos y sujetos a requisitos y controles.', False], ['LEY32069-04', 'Ley N.° 32069', 'Ejecución contractual', 'Resumen didáctico simulado: luego de la adjudicación, la ejecución contractual debe respetar obligaciones, plazos, prestaciones y mecanismos de control.', False], ['LEY32069-05', 'Ley N.° 32069', 'Responsabilidades del proveedor', 'Resumen didáctico simulado: los proveedores deben cumplir las obligaciones asumidas y responder por incumplimientos conforme al régimen aplicable.', False], ['LEY32069-06', 'Ley N.° 32069', 'Transparencia y trazabilidad', 'Resumen didáctico simulado: el proceso de contratación debe conservar información suficiente para permitir transparencia, revisión y trazabilidad.', False], ['PDUPDM-01', 'PDU/PDM Huancayo-El Tambo', 'Zonificación urbana', 'Resumen didáctico simulado: la zonificación clasifica áreas del territorio y orienta los usos permitidos, condicionados o restringidos en cada zona.', False], ['PDUPDM-02', 'PDU/PDM Huancayo-El Tambo', 'Compatibilidad de uso de suelo', 'Resumen didáctico simulado: si el uso propuesto para un proyecto es incompatible con la zonificación vigente, el expediente requiere observación y revisión jurídica.', True], ['PDUPDM-03', 'PDU/PDM Huancayo-El Tambo', 'Parámetros urbanísticos', 'Resumen didáctico simulado: los proyectos urbanos deben revisar parámetros aplicables como ocupación, altura, retiros u otras condiciones definidas por el instrumento territorial.', False], ['PDUPDM-04', 'PDU/PDM Huancayo-El Tambo', 'Zonas de riesgo', 'Resumen didáctico simulado: una propuesta ubicada en un área identificada como riesgo alto o no mitigable debe ser observada y sometida a revisión antes de continuar.', True], ['PDUPDM-05', 'PDU/PDM Huancayo-El Tambo', 'Sistema vial', 'Resumen didáctico simulado: la evaluación territorial debe considerar jerarquía vial, accesibilidad y posibles afectaciones a la red vial planificada.', False], ['PDUPDM-06', 'PDU/PDM Huancayo-El Tambo', 'Vigencia y actualización territorial', 'Resumen didáctico simulado: para evaluar un proyecto debe utilizarse la versión vigente del instrumento territorial y registrarse la fuente o versión consultada.', False]]

df = pd.DataFrame(
    records,
    columns=["id", "documento", "tema", "contenido", "alerta"]
)

print("Fragmentos:", len(df))
display(df.head(10))
df.to_csv("base_normativa_simulada_poc.csv", index=False, encoding="utf-8-sig")
print("CSV generado: base_normativa_simulada_poc.csv")


Fragmentos: 24


,id,documento,tema,contenido,alerta
0,LEY27972-01,Ley N.° 27972,Competencias municipales,Resumen didáctico simulado: las municipalidades ejercen competencias y funciones en asuntos de interés local dentro ...,False
1,LEY27972-02,Ley N.° 27972,Desarrollo urbano,"Resumen didáctico simulado: la gestión municipal comprende funciones vinculadas con organización del espacio físico,...",False
2,LEY27972-03,Ley N.° 27972,Concejo municipal,Resumen didáctico simulado: el concejo municipal cumple funciones normativas y fiscalizadoras dentro del gobierno lo...,False
3,LEY27972-04,Ley N.° 27972,Alcaldía,Resumen didáctico simulado: la alcaldía ejerce funciones ejecutivas y de representación del gobierno local conforme ...,False
4,LEY27972-05,Ley N.° 27972,Servicios públicos locales,Resumen didáctico simulado: la municipalidad participa en la organización y prestación de servicios públicos locales...,False
5,LEY27972-06,Ley N.° 27972,Planeamiento y presupuesto,"Resumen didáctico simulado: la gestión municipal articula instrumentos de planeamiento, presupuesto y programación p...",False
6,DL1252-01,D. L. N.° 1252 - Invierte.pe,Programación multianual,Resumen didáctico simulado: la programación multianual orienta la cartera de inversiones hacia el cierre de brechas ...,False
7,DL1252-02,D. L. N.° 1252 - Invierte.pe,Formulación y evaluación,Resumen didáctico simulado: los proyectos pasan por actividades de formulación y evaluación para sustentar su conven...,False
8,DL1252-03,D. L. N.° 1252 - Invierte.pe,Ejecución de inversiones,Resumen didáctico simulado: la ejecución comprende el desarrollo de las inversiones aprobadas conforme a la programa...,False
9,DL1252-04,D. L. N.° 1252 - Invierte.pe,Cierre de brechas,Resumen didáctico simulado: el sistema de inversiones públicas busca orientar recursos hacia brechas de infraestruct...,False


CSV generado: base_normativa_simulada_poc.csv



## 2. Conjunto de evaluación

Se utilizan **30 consultas** con una respuesta esperada (`id_esperado`).  
Esto permite calcular métricas objetivas y no depender solo de una impresión visual.


In [3]:

preguntas = [('¿Qué marco se relaciona con las competencias y funciones del gobierno local?', 'LEY27972-01'), ('¿Qué fuente debería consultar para funciones municipales sobre desarrollo urbano?', 'LEY27972-02'), ('¿Qué órgano municipal cumple funciones normativas y fiscalizadoras?', 'LEY27972-03'), ('¿Qué parte del gobierno local ejerce funciones ejecutivas y de representación?', 'LEY27972-04'), ('¿Qué fuente se relaciona con la prestación de servicios públicos locales?', 'LEY27972-05'), ('¿Qué fragmento trata sobre planeamiento y presupuesto municipal?', 'LEY27972-06'), ('¿Dónde se aborda la programación multianual de inversiones?', 'DL1252-01'), ('¿Qué etapa sustenta la viabilidad antes de continuar con una inversión?', 'DL1252-02'), ('¿Qué fragmento trata sobre la ejecución de inversiones aprobadas?', 'DL1252-03'), ('¿Qué fuente relaciona las inversiones con el cierre de brechas?', 'DL1252-04'), ('¿Qué fragmento explica la organización de una cartera de inversiones?', 'DL1252-05'), ('¿Qué parte se relaciona con seguimiento de avances de inversiones?', 'DL1252-06'), ('¿Qué fragmento menciona integridad, competencia y transparencia en contratación pública?', 'LEY32069-01'), ('¿Qué información trata sobre planificación de contrataciones según necesidad y presupuesto?', 'LEY32069-02'), ('¿Dónde se describen procedimientos para seleccionar proveedores?', 'LEY32069-03'), ('¿Qué fragmento se refiere al cumplimiento de obligaciones y plazos después de adjudicar?', 'LEY32069-04'), ('¿Qué parte se relaciona con responsabilidades del proveedor por incumplimiento?', 'LEY32069-05'), ('¿Qué fragmento enfatiza transparencia y trazabilidad del proceso de contratación?', 'LEY32069-06'), ('¿Qué instrumento permite identificar la zonificación urbana de un área?', 'PDUPDM-01'), ('¿Qué fragmento debo consultar para saber si un uso es compatible con la zonificación?', 'PDUPDM-02'), ('¿Dónde se revisan parámetros como altura, retiros u ocupación?', 'PDUPDM-03'), ('¿Qué fragmento sirve para revisar proyectos ubicados en zonas de riesgo?', 'PDUPDM-04'), ('¿Qué parte del instrumento territorial habla de jerarquía y accesibilidad vial?', 'PDUPDM-05'), ('¿Qué fragmento exige trabajar con la versión vigente del instrumento territorial?', 'PDUPDM-06'), ('Un proyecto municipal necesita comprobar si la actividad propuesta está permitida en el terreno. ¿Qué debe consultarse?', 'PDUPDM-02'), ('Un expediente está situado en un área de riesgo alto. ¿Qué evidencia territorial debería recuperarse?', 'PDUPDM-04'), ('Para priorizar inversiones según brechas de servicios, ¿qué fragmento de Invierte.pe es relevante?', 'DL1252-04'), ('Antes de ejecutar una inversión, ¿qué etapa analiza y sustenta su viabilidad?', 'DL1252-02'), ('Para auditar la trazabilidad de una contratación, ¿qué fragmento debería recuperarse?', 'LEY32069-06'), ('Para justificar una competencia del gobierno local, ¿qué fragmento de la Ley Orgánica de Municipalidades debería recuperarse?', 'LEY27972-01')]
eval_df = pd.DataFrame(preguntas, columns=["pregunta", "id_esperado"])
print("Consultas de evaluación:", len(eval_df))
display(eval_df.head())


Consultas de evaluación: 30


,pregunta,id_esperado
0,¿Qué marco se relaciona con las competencias y funciones del gobierno local?,LEY27972-01
1,¿Qué fuente debería consultar para funciones municipales sobre desarrollo urbano?,LEY27972-02
2,¿Qué órgano municipal cumple funciones normativas y fiscalizadoras?,LEY27972-03
3,¿Qué parte del gobierno local ejerce funciones ejecutivas y de representación?,LEY27972-04
4,¿Qué fuente se relaciona con la prestación de servicios públicos locales?,LEY27972-05



## 3. Búsqueda léxica BM25

BM25 aporta coincidencia por palabras. Después se combinará con la búsqueda semántica para aproximar el **RAG híbrido** definido en la US02.


In [4]:

def tokenizar(texto):
    return re.findall(r"\b\w+\b", texto.lower(), flags=re.UNICODE)

bm25_corpus = [tokenizar(x) for x in df["contenido"].tolist()]
bm25 = BM25Okapi(bm25_corpus)

def buscar_bm25(consulta, top_k=5):
    scores = bm25.get_scores(tokenizar(consulta))
    idx = np.argsort(scores)[::-1][:top_k]
    return [
        {"id": df.iloc[i]["id"], "score_bm25": float(scores[i]), "index": int(i)}
        for i in idx
    ]

buscar_bm25("compatibilidad del uso del suelo con la zonificación", 3)


[{'id': 'PDUPDM-02', 'score_bm25': 6.027566165584541, 'index': 19},
 {'id': 'PDUPDM-01', 'score_bm25': 4.8328869543846285, 'index': 18},
 {'id': 'LEY27972-02', 'score_bm25': 4.255321605586552, 'index': 1}]


## 4. Embeddings y Qdrant

Se crea una colección **Qdrant en memoria** para cada modelo.  
No necesitas cuenta de Qdrant Cloud para esta PoC.

Los modelos locales se descargan desde Hugging Face al ejecutarlos por primera vez.


In [5]:

MODELOS_LOCALES = {
    "BGE-M3": "BAAI/bge-m3",
    "Qwen3-Embedding-0.6B": "Qwen/Qwen3-Embedding-0.6B",
}

def cargar_modelo_local(model_id):
    print(f"Cargando {model_id} ...")
    return SentenceTransformer(model_id)

def crear_indice_qdrant(modelo, nombre_modelo):
    textos = df["contenido"].tolist()
    embeddings = modelo.encode(
        textos,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    dim = embeddings.shape[1]

    client = QdrantClient(":memory:")
    collection = "normativa"

    client.create_collection(
        collection_name=collection,
        vectors_config=models.VectorParams(
            size=dim,
            distance=models.Distance.COSINE
        ),
    )

    points = []
    for i, emb in enumerate(embeddings):
        row = df.iloc[i]
        points.append(
            models.PointStruct(
                id=i,
                vector=emb.tolist(),
                payload={
                    "id_norma": row["id"],
                    "documento": row["documento"],
                    "tema": row["tema"],
                    "contenido": row["contenido"],
                    "alerta": bool(row["alerta"])
                }
            )
        )

    client.upsert(collection_name=collection, points=points)
    print(f"{nombre_modelo}: {len(points)} fragmentos indexados en Qdrant.")
    return client

def buscar_semantico(client, modelo, consulta, top_k=5):
    q = modelo.encode([consulta], normalize_embeddings=True)[0].tolist()

    # Compatible con versiones recientes de qdrant-client
    hits = client.query_points(
        collection_name="normativa",
        query=q,
        limit=top_k,
        with_payload=True
    ).points

    return [
        {
            "id": h.payload["id_norma"],
            "score_dense": float(h.score),
            "payload": h.payload
        }
        for h in hits
    ]



## 5. Fusión híbrida: Dense + BM25

Se utiliza **Reciprocal Rank Fusion (RRF)** para combinar ambos rankings.  
Esto evita depender de que las escalas de puntuación de BM25 y coseno sean iguales.


In [6]:

def buscar_hibrido(client, modelo, consulta, top_k=5, candidatos=10, k_rrf=60):
    dense = buscar_semantico(client, modelo, consulta, top_k=candidatos)
    lexical = buscar_bm25(consulta, top_k=candidatos)

    score_rrf = {}
    payloads = {}

    for rank, item in enumerate(dense, start=1):
        score_rrf[item["id"]] = score_rrf.get(item["id"], 0) + 1/(k_rrf + rank)
        payloads[item["id"]] = item.get("payload")

    for rank, item in enumerate(lexical, start=1):
        score_rrf[item["id"]] = score_rrf.get(item["id"], 0) + 1/(k_rrf + rank)

    orden = sorted(score_rrf.items(), key=lambda x: x[1], reverse=True)[:top_k]

    resultados = []
    for id_norma, score in orden:
        fila = df[df["id"] == id_norma].iloc[0]
        resultados.append({
            "id": id_norma,
            "documento": fila["documento"],
            "tema": fila["tema"],
            "contenido": fila["contenido"],
            "alerta": bool(fila["alerta"]),
            "score_rrf": score
        })
    return resultados



## 6. Métricas

El documento del proyecto fija una meta de **precisión/relevancia superior o igual a 94 %**.

Para volver esa meta medible en esta PoC se reportan:

- **Accuracy@1**: el primer fragmento recuperado es exactamente el esperado.
- **Hit@3**: el fragmento esperado aparece dentro de los tres primeros resultados.
- **MRR**: premia que el resultado correcto aparezca en posiciones altas.
- **Latencia promedio** por consulta.

**Criterio operativo de esta notebook:** `Hit@3 >= 0.94`.

> Si el docente decide que el 94 % debe referirse a otra métrica, solo hay que cambiar el criterio de decisión; los resultados completos quedan registrados.


In [7]:

def evaluar_modelo(nombre, modelo, client):
    filas = []
    tiempos = []

    for _, row in eval_df.iterrows():
        inicio = time.perf_counter()
        res = buscar_hibrido(client, modelo, row["pregunta"], top_k=3)
        tiempos.append(time.perf_counter() - inicio)

        ids = [x["id"] for x in res]
        esperado = row["id_esperado"]

        rank = ids.index(esperado) + 1 if esperado in ids else None
        filas.append({
            "modelo": nombre,
            "pregunta": row["pregunta"],
            "esperado": esperado,
            "top1": ids[0] if ids else None,
            "top3": ", ".join(ids),
            "acierto_top1": int(len(ids) > 0 and ids[0] == esperado),
            "hit_top3": int(esperado in ids),
            "rr": 1/rank if rank else 0
        })

    detalle = pd.DataFrame(filas)
    resumen = {
        "Modelo": nombre,
        "Accuracy@1": detalle["acierto_top1"].mean(),
        "Hit@3": detalle["hit_top3"].mean(),
        "MRR": detalle["rr"].mean(),
        "Latencia_promedio_s": np.mean(tiempos),
        "Cumple_94_Hit3": detalle["hit_top3"].mean() >= 0.94
    }
    return detalle, resumen



## 7. Ejecutar los dos modelos locales

> **Aviso de memoria:** BGE-M3 es el modelo más pesado. Si Colab se queda sin RAM/VRAM, ejecuta primero Qwen3, guarda el resultado, reinicia el entorno y luego ejecuta BGE-M3.


In [8]:

resultados_detalle = {}
resumen_modelos = []

for nombre, model_id in MODELOS_LOCALES.items():
    print("\n" + "="*80)
    print("MODELO:", nombre)
    modelo = cargar_modelo_local(model_id)
    client = crear_indice_qdrant(modelo, nombre)
    detalle, resumen = evaluar_modelo(nombre, modelo, client)
    resultados_detalle[nombre] = detalle
    resumen_modelos.append(resumen)
    print(resumen)

resumen_df = pd.DataFrame(resumen_modelos)
display(resumen_df)



MODELO: BGE-M3
Cargando BAAI/bge-m3 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

... (26 outputs left out)

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Qwen3-Embedding-0.6B: 24 fragmentos indexados en Qdrant.
{'Modelo': 'Qwen3-Embedding-0.6B', 'Accuracy@1': np.float64(0.9666666666666667), 'Hit@3': np.float64(1.0), 'MRR': np.float64(0.9833333333333333), 'Latencia_promedio_s': np.float64(0.07056058850000682), 'Cumple_94_Hit3': np.True_}


,Modelo,Accuracy@1,Hit@3,MRR,Latencia_promedio_s,Cumple_94_Hit3
0,BGE-M3,0.900000,1.0,0.944444,0.031201,True
1,Qwen3-Embedding-0.6B,0.966667,1.0,0.983333,0.070561,True



## 8. Modelo OpenAI opcional: `text-embedding-3-small`

Esta parte es **opcional** porque requiere una API key y puede generar costo.

Si no deseas utilizar una API externa, omite esta sección.  
La PoC seguirá siendo ejecutable con los dos modelos locales.


In [9]:

USAR_OPENAI = False  # Cambia a True si deseas probar text-embedding-3-small

if USAR_OPENAI:
    from openai import OpenAI
    api_key = getpass.getpass("OPENAI_API_KEY: ")
    oa = OpenAI(api_key=api_key)

    def openai_embed(texts):
        resp = oa.embeddings.create(
            model="text-embedding-3-small",
            input=texts
        )
        return np.array([d.embedding for d in resp.data], dtype=np.float32)

    # Se deja preparado el método. Para mantener la notebook simple,
    # la comparación obligatoria se realiza con BGE-M3 y Qwen3.
    print("OpenAI configurado. Puedes extender crear_indice_qdrant() para usar openai_embed().")
else:
    print("OpenAI omitido.")


OpenAI omitido.



## 9. Demostración de la US02

Esta función simula una consulta del **Asesor Jurídico**.  
Devuelve la evidencia normativa recuperada y marca **ALERTA LEGAL DE PoC** si alguno de los fragmentos recuperados está etiquetado como alerta.

La alerta es **solo técnica/didáctica** y no constituye una conclusión legal real.


In [10]:

# Escoge el modelo que quieras demostrar.
NOMBRE_DEMO = "Qwen3-Embedding-0.6B"
MODEL_ID_DEMO = MODELOS_LOCALES[NOMBRE_DEMO]

modelo_demo = cargar_modelo_local(MODEL_ID_DEMO)
client_demo = crear_indice_qdrant(modelo_demo, NOMBRE_DEMO)

def consultar_agente_juridico_poc(consulta):
    res = buscar_hibrido(client_demo, modelo_demo, consulta, top_k=3)

    print("CONSULTA:")
    print(consulta)
    print("\nEVIDENCIA RECUPERADA:")
    for i, r in enumerate(res, 1):
        print(f"\n{i}. {r['id']} | {r['documento']} | {r['tema']}")
        print(r["contenido"])

    alerta = any(r["alerta"] for r in res[:2])
    print("\nRESULTADO DE PoC:")
    print("ALERTA LEGAL — REQUIERE REVISIÓN HUMANA" if alerta else
          "SIN ALERTA AUTOMÁTICA EN ESTA PoC — REQUIERE VALIDACIÓN HUMANA")

    return res

consulta_demo = (
    "Se propone construir un equipamiento municipal en un terreno cuyo uso podría "
    "ser incompatible con la zonificación vigente. ¿Qué evidencia debe revisar el Agente Jurídico?"
)

_ = consultar_agente_juridico_poc(consulta_demo)


Cargando Qwen/Qwen3-Embedding-0.6B ...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Qwen3-Embedding-0.6B: 24 fragmentos indexados en Qdrant.
CONSULTA:
Se propone construir un equipamiento municipal en un terreno cuyo uso podría ser incompatible con la zonificación vigente. ¿Qué evidencia debe revisar el Agente Jurídico?

EVIDENCIA RECUPERADA:

1. PDUPDM-02 | PDU/PDM Huancayo-El Tambo | Compatibilidad de uso de suelo
Resumen didáctico simulado: si el uso propuesto para un proyecto es incompatible con la zonificación vigente, el expediente requiere observación y revisión jurídica.

2. PDUPDM-04 | PDU/PDM Huancayo-El Tambo | Zonas de riesgo
Resumen didáctico simulado: una propuesta ubicada en un área identificada como riesgo alto o no mitigable debe ser observada y sometida a revisión antes de continuar.

3. PDUPDM-01 | PDU/PDM Huancayo-El Tambo | Zonificación urbana
Resumen didáctico simulado: la zonificación clasifica áreas del territorio y orienta los usos permitidos, condicionados o restringidos en cada zona.

RESULTADO DE PoC:
ALERTA LEGAL — REQUIERE REVISIÓN HUMANA


## 10. Decisión GO / NO-GO

Regla utilizada para esta PoC:

- **GO:** al menos un modelo logra `Hit@3 >= 94 %`.
- **GO CON CONDICIONES:** el mejor modelo queda entre `85 %` y `< 94 %`, o existen limitaciones importantes que deben corregirse.
- **NO-GO:** ningún modelo llega a `85 %`.

La decisión se genera con los **resultados reales de tu ejecución**, no con cifras inventadas.


In [11]:

mejor = resumen_df.sort_values("Hit@3", ascending=False).iloc[0]
hit = float(mejor["Hit@3"])

if hit >= 0.94:
    decision = "GO"
elif hit >= 0.85:
    decision = "GO CON CONDICIONES"
else:
    decision = "NO-GO"

print("===== RESULTADO FINAL DE LA PoC =====")
print("Mejor modelo:", mejor["Modelo"])
print(f"Accuracy@1: {mejor['Accuracy@1']:.2%}")
print(f"Hit@3:      {mejor['Hit@3']:.2%}")
print(f"MRR:        {mejor['MRR']:.3f}")
print(f"Latencia:   {mejor['Latencia_promedio_s']:.3f} s")
print("DECISIÓN:", decision)

resumen_df.to_csv("resultados_resumen_poc.csv", index=False, encoding="utf-8-sig")

for nombre, detalle in resultados_detalle.items():
    safe = re.sub(r"[^A-Za-z0-9_-]+", "_", nombre)
    detalle.to_csv(f"detalle_{safe}.csv", index=False, encoding="utf-8-sig")

print("\nArchivos generados:")
print("- resultados_resumen_poc.csv")
print("- detalle_<modelo>.csv")


===== RESULTADO FINAL DE LA PoC =====
Mejor modelo: BGE-M3
Accuracy@1: 90.00%
Hit@3:      100.00%
MRR:        0.944
Latencia:   0.031 s
DECISIÓN: GO

Archivos generados:
- resultados_resumen_poc.csv
- detalle_<modelo>.csv



## 11. Qué copiar al documento 6 después de ejecutar

Guarda como evidencia:

1. captura de la tabla `resumen_df`;
2. captura de una consulta de demostración;
3. resultado final **GO / GO CON CONDICIONES / NO-GO**;
4. los CSV generados;
5. enlace o archivo de esta notebook.

### Texto que puedes completar con tus resultados

> Se implementó una PoC en Google Colab utilizando Python, Qdrant en memoria, BM25 y modelos de embeddings. Se ejecutaron 30 consultas etiquetadas sobre una base normativa de prueba. El mejor modelo fue **[MODELO]**, con un Accuracy@1 de **[X %]**, Hit@3 de **[X %]**, MRR de **[X]** y latencia promedio de **[X s]**. Con relación al criterio operativo de relevancia ≥ 94 %, la decisión fue **[GO / GO CON CONDICIONES / NO-GO]**.

### Limitación que debes mencionar

La base incluida en esta notebook es **simulada**. Antes de presentar la validación jurídica como evidencia final, deben reemplazarse los resúmenes por fragmentos obtenidos de las **versiones oficiales y vigentes** de la normativa y del PDU/PDM.
